# Train the ablations (A1, A2, A3)

A1 experts trainable in Stage C, A2 no class-conditional reliability, A3 private encoders. A1 and A2 reuse M's Stage A and B (only Stage C is trained); A3 starts every encoder from M's Stage A and trains Stages B and C.

* **Runtime:** about TODO hours per condition and seed on a T4/L4 GPU (see docs/REPRODUCING.md)
* **GPU:** required (Runtime > Change runtime type > GPU)
* **Before running:** follow [docs/DATA.md](https://github.com/selimsidan/dmoe_nids/blob/main/docs/DATA.md) to put the four NF-v3 CSVs in your Drive (or set `DATA_ROOT`).
* **How to use:** edit only the SETTINGS cell, then *Runtime > Run all*. Rerunning after a disconnect continues where training stopped.
* **Needs:** M trained for the same seeds (notebook 01).


In [ ]:
# ============== SETTINGS: the only cell you edit ==============
REPO_REF    = "main"     # branch or release tag (e.g. "v1.0.0") of the repository
STORAGE     = "drive"    # "drive": Google Drive (persists) | "local": /content (lost at disconnect)
DATA_ROOT   = None       # None: Colab secret NIDS_DATA_DIR, else MyDrive/NIDS_datasets
OUTPUT_ROOT = None       # None: Colab secret NIDS_OUTPUT_DIR, else MyDrive/NIDS_outputs
# Experiment
CONDITIONS = ['A1', 'A2', 'A3']   # subset allowed, e.g. ['A1']
SEEDS      = [0, 1, 2, 3, 4]   # any list, e.g. [3, 4]
EXECUTE    = False   # False: print the plan only. True: train.
RUNS_PER_SESSION = 1   # (condition, seed) runs per execution; None = until done or disconnected
# ===============================================================

In [ ]:
# Setup: clone the repository, install it, mount storage and resolve the roots.
import os, subprocess, sys

REPO_URL, TARGET = "https://github.com/selimsidan/dmoe_nids.git", "/content/dmoe_nids"

def _git(*args, cwd=None, env=None):
    subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True,
                   env={**os.environ, **(env or {})})

def _fetch(env=None):
    if os.path.isdir(os.path.join(TARGET, ".git")):
        _git("fetch", "--quiet", "--tags", "origin", cwd=TARGET, env=env)
    else:
        _git("clone", "--quiet", REPO_URL, TARGET, env=env)

try:
    _fetch()
except subprocess.CalledProcessError:
    # Only for a private repository or fork: retry with the Colab secret GITHUB_TOKEN (never printed).
    import base64
    from google.colab import userdata
    _token = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    _fetch({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {_token}"})
    del _token
_git("checkout", "--quiet", REPO_REF, cwd=TARGET)
if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=TARGET).returncode == 0:
    # Fast-forward to the commits fetched above (no second download, so no token needed).
    _git("merge", "--quiet", "--ff-only", f"origin/{REPO_REF}", cwd=TARGET)

# Installs the package and any missing dependencies; Colab's own torch build is kept.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", TARGET], check=True)
sys.path.insert(0, os.path.join(TARGET, "src"))
os.environ["PYTHONPATH"] = os.path.join(TARGET, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(TARGET)

import dmoe.colab
ROOTS = dmoe.colab.setup(STORAGE, DATA_ROOT, OUTPUT_ROOT)

In [ ]:
# Preflight: GPU, data files, and the plan (completed runs are skipped).
from dmoe.colab import gpu_name
from dmoe.data.verify import verify_data
from dmoe.training.campaign import load_campaign, plan_units, print_plan

if gpu_name() is None:
    print('WARNING: no GPU. Training on CPU is impractically slow; switch the runtime type.')
checks = verify_data(ROOTS['data'].path, count=False, hash_files=False)  # existence and header
assert all(c.ok for c in checks), 'fix the data problems above (docs/DATA.md)'
CAMPAIGN = load_campaign()
print_plan(plan_units(CAMPAIGN, CONDITIONS, SEEDS, ROOTS['output'].path))

In [ ]:
# Train. Output streams here and into runs/<condition>/seed<k>/log.txt on your storage.
from dmoe.training.campaign import run_campaign

units = run_campaign(CAMPAIGN, CONDITIONS, SEEDS, execute=EXECUTE,
                     max_units=RUNS_PER_SESSION, output_root=ROOTS['output'].path)

In [ ]:
# Results of the completed runs (test split). Cross-seed tables: notebook 04, MODE='tables'.
import os, pandas as pd
from IPython.display import Image, display

rows = []
for condition in CONDITIONS:
    for seed in SEEDS:
        path = os.path.join(ROOTS['output'].path, 'runs', condition, f'seed{seed}', 'eval', 'test', 'Overall_Metrics.csv')
        if os.path.isfile(path):
            frame = pd.read_csv(path)
            rows.append(frame[frame['origin'] == 'ALL'])
if rows:
    table = pd.concat(rows)[['condition', 'seed', 'macro_f1', 'accuracy', 'balanced_accuracy', 'weighted_f1']]
    display(table.reset_index(drop=True))
    last = table.iloc[-1]
    figure = os.path.join(ROOTS['output'].path, 'runs', last['condition'], f"seed{int(last['seed'])}", 'eval', 'test', 'figures', 'Confusion_Matrix_ALL.png')
    if os.path.isfile(figure):
        display(Image(figure, width=900))
else:
    print('No completed runs yet for these settings.')